# Dataset Statistics

In [1]:
import os
import numpy as np
import pandas as pd

# -----------------------------
# 0) Load FULL final dataset
# -----------------------------
DATA_PATH = "merged_sample_one_processed/final_stage"  # folder, not a single part
df = pd.read_parquet(DATA_PATH)

print("✅ Loaded final merged dataset")
print("Shape (rows, cols):", df.shape)

# -----------------------------
# 1) Make statistics folder
# -----------------------------
stats_dir = "statistics"
os.makedirs(stats_dir, exist_ok=True)

# -----------------------------
# 2) Core dataset summary table
# -----------------------------
target_col   = "susceptibility"
patient_col  = "anon_id"
enc_col      = "pat_enc_csn_id_coded"   # change if yours differs

summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "n_patients": df[patient_col].nunique() if patient_col in df.columns else np.nan,
    "n_encounters": df[enc_col].nunique() if enc_col in df.columns else np.nan,
}

summary_df = pd.DataFrame([summary])

# SHOW in notebook
display(summary_df)

# SAVE
summary_path = os.path.join(stats_dir, "dataset_statistics.csv")
summary_df.to_csv(summary_path, index=False)
print(f"💾 Saved: {summary_path}")

# -----------------------------
# 3) Target distribution (counts + %)
# -----------------------------
if target_col in df.columns:
    td = df[target_col].value_counts(dropna=False).rename("count").to_frame()
    td["percent"] = (td["count"] / td["count"].sum()) * 100.0

    # SHOW
    display(td)

    # SAVE
    td_path = os.path.join(stats_dir, "target_distribution.csv")
    td.to_csv(td_path)
    print(f"💾 Saved: {td_path}")
else:
    print(f"⚠ target_col '{target_col}' not found in df.columns")

# -----------------------------
# 4) Missingness table (Top 30)
# -----------------------------
missing = df.isna().sum().sort_values(ascending=False)
missing_df = pd.DataFrame({
    "feature": missing.index.astype(str),
    "missing_count": missing.values.astype(int),
})
missing_df["missing_percent"] = (missing_df["missing_count"] / df.shape[0]) * 100.0

top_missing = missing_df.head(30)

# SHOW
display(top_missing)

# SAVE
missing_path = os.path.join(stats_dir, "missingness_top30.csv")
top_missing.to_csv(missing_path, index=False)
print(f"💾 Saved: {missing_path}")

# -----------------------------
# 5) Rows-per-patient summary (answers “1000 patients → millions of rows”)
# -----------------------------
if patient_col in df.columns:
    rows_per_patient = df.groupby(patient_col).size()

    rpp_summary = pd.DataFrame([{
        "rows_per_patient_mean": float(rows_per_patient.mean()),
        "rows_per_patient_median": float(rows_per_patient.median()),
        "rows_per_patient_p95": float(rows_per_patient.quantile(0.95)),
        "rows_per_patient_p99": float(rows_per_patient.quantile(0.99)),
        "rows_per_patient_max": float(rows_per_patient.max()),
    }])

    # SHOW
    display(rpp_summary)

    # SAVE
    rpp_path = os.path.join(stats_dir, "rows_per_patient_summary.csv")
    rpp_summary.to_csv(rpp_path, index=False)
    print(f"💾 Saved: {rpp_path}")

    # Optional: also show top 10 “super patients”
    top10 = rows_per_patient.sort_values(ascending=False).head(10).rename("rows").to_frame()
    display(top10)
    top10_path = os.path.join(stats_dir, "top10_patients_by_rows.csv")
    top10.to_csv(top10_path)
    print(f"💾 Saved: {top10_path}")
else:
    print(f"⚠ patient_col '{patient_col}' not found in df.columns")

✅ Loaded final merged dataset
Shape (rows, cols): (1213641, 71)


,n_rows,n_columns,n_patients,n_encounters
0,1213641,71,997,1929


💾 Saved: statistics\dataset_statistics.csv


,count,percent
susceptibility,,
0,827853,68.212346
1,385788,31.787654


💾 Saved: statistics\target_distribution.csv


...

💾 Saved: statistics\missingness_top30.csv


,rows_per_patient_mean,rows_per_patient_median,rows_per_patient_p95,rows_per_patient_p99,rows_per_patient_max
0,1217.292879,74.0,4422.8,16159.2,237336.0


💾 Saved: statistics\rows_per_patient_summary.csv


,rows
anon_id,
JC6241803,237336
JC1235629,135784
JC1214065,42744
JC674480,41325
JC2814824,25755
JC1235833,21803
JC2092152,18223
JC517943,18143
JC2731156,17136


💾 Saved: statistics\top10_patients_by_rows.csv


In [2]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

import pandas as pd

df = pd.read_parquet("merged_sample_one_processed/final_stage/part.0.parquet")
print(f"Shape: {df.shape}")

def _is_dask_df(x):
    return x.__class__.__module__.startswith("dask.dataframe")

def _nunique(series):
    if _is_dask_df(series.to_frame()):
        return int(series.nunique().compute())
    return int(series.nunique(dropna=True))

def _value_counts(series, dropna=False):
    if _is_dask_df(series.to_frame()):
        vc = series.value_counts(dropna=dropna).compute()
        return vc
    return series.value_counts(dropna=dropna)

def dataset_statistics_table(
    df,
    out_dir="statistics",
    target_col="susceptibility",
    patient_col="anon_id",
    encounter_col="pat_enc_csn_id_coded",
    culture_keys=None,
    top_missing_k=30,
):
    os.makedirs(out_dir, exist_ok=True)

    is_dask = _is_dask_df(df)

    # ------------------------
    # Basic shape
    # ------------------------
    n_rows = int(df.shape[0].compute()) if is_dask else int(df.shape[0])
    n_cols = int(df.shape[1])

    has_patient = patient_col in df.columns
    has_enc = encounter_col in df.columns

    n_patients = _nunique(df[patient_col]) if has_patient else np.nan
    n_encounters = _nunique(df[encounter_col]) if has_enc else np.nan

    # ------------------------
    # Unique cultures
    # ------------------------
    n_cultures = np.nan
    if culture_keys:
        missing_keys = [c for c in culture_keys if c not in df.columns]
        if len(missing_keys) == 0:
            if is_dask:
                n_cultures = int(df[culture_keys].drop_duplicates().shape[0].compute())
            else:
                n_cultures = int(df[culture_keys].drop_duplicates().shape[0])

    # ------------------------
    # Target distribution
    # ------------------------
    if target_col in df.columns:
        vc = _value_counts(df[target_col], dropna=False)
        vc_df = vc.rename("count").to_frame()
        vc_df["percent"] = (vc_df["count"] / vc_df["count"].sum()) * 100.0

        vc_df.to_csv(os.path.join(out_dir, "target_distribution.csv"))

        print("\n📊 Target Distribution")
        display(vc_df)

        prevalence = np.nan
        if "Resistant" in vc.index:
            prevalence = float(vc["Resistant"] / vc.sum())
    else:
        vc_df = None
        prevalence = np.nan

    # ------------------------
    # Missingness
    # ------------------------
    if is_dask:
        na_counts = df.isna().sum().compute()
    else:
        na_counts = df.isna().sum()

    missing_df = (
        pd.DataFrame({
            "feature": na_counts.index.astype(str),
            "missing_count": na_counts.values.astype(int),
        })
        .assign(missing_percent=lambda x: 100.0 * x["missing_count"] / n_rows)
        .sort_values("missing_percent", ascending=False)
        .reset_index(drop=True)
    )

    top_missing = missing_df.head(top_missing_k)

    top_missing.to_csv(
        os.path.join(out_dir, f"missingness_top{top_missing_k}.csv"),
        index=False
    )

    print(f"\n📊 Top {top_missing_k} Features by Missingness")
    display(top_missing)

    # ------------------------
    # Rows per patient
    # ------------------------
    rows_per_patient_path = None
    if has_patient:
        if is_dask:
            rpp = df.groupby(patient_col).size().compute()
        else:
            rpp = df.groupby(patient_col).size()

        rpp = rpp.astype(int)

        rows_per_patient_stats = pd.DataFrame([{
            "rows_per_patient_mean": float(rpp.mean()),
            "rows_per_patient_median": float(rpp.median()),
            "rows_per_patient_p95": float(rpp.quantile(0.95)),
            "rows_per_patient_p99": float(rpp.quantile(0.99)),
            "rows_per_patient_max": float(rpp.max()),
        }])

        rows_per_patient_path = os.path.join(out_dir, "rows_per_patient_summary.csv")
        rows_per_patient_stats.to_csv(rows_per_patient_path, index=False)

        print("\n📊 Rows per Patient Summary")
        display(rows_per_patient_stats)

    # ------------------------
    # Main dataset statistics
    # ------------------------
    stats = pd.DataFrame([{
        "n_rows": n_rows,
        "n_columns": n_cols,
        "n_patients": n_patients,
        "n_encounters": n_encounters,
        "n_unique_cultures": n_cultures,
        "target_col": target_col if target_col in df.columns else np.nan,
        "prevalence_resistant": prevalence,
        "missing_features_gt_50pct": int((missing_df["missing_percent"] > 50).sum()),
        "missing_features_gt_80pct": int((missing_df["missing_percent"] > 80).sum()),
    }])

    main_path = os.path.join(out_dir, "dataset_statistics.csv")
    stats.to_csv(main_path, index=False)

    print("\n📊 Dataset Summary")
    display(stats)

    # ------------------------
    # Extra patient/encounter counts
    # ------------------------
    if has_patient or has_enc:
        ge = pd.DataFrame([{
            "n_patients": n_patients,
            "n_encounters": n_encounters,
            "patient_col": patient_col if has_patient else np.nan,
            "encounter_col": encounter_col if has_enc else np.nan,
        }])

        ge_path = os.path.join(out_dir, "group_counts_patients_encounters.csv")
        ge.to_csv(ge_path, index=False)

        print("\n📊 Patient / Encounter Counts")
        display(ge)

    print("\n✅ Files saved to:", out_dir)

    return stats, missing_df, vc_df


# Example usage
culture_keys = ["anon_id","pat_enc_csn_id_coded","order_proc_id_coded","order_time_jittered_utc"]

stats, missing_df, target_df = dataset_statistics_table(
    df,
    out_dir="statistics",
    target_col="susceptibility",
    patient_col="anon_id",
    encounter_col="pat_enc_csn_id_coded",
    culture_keys=culture_keys,
    top_missing_k=30
)

stats

Shape: (1213641, 71)

📊 Target Distribution


,count,percent
susceptibility,,
0,827853,68.212346
1,385788,31.787654



📊 Top 30 Features by Missingness


,feature,missing_count,missing_percent
0,Q75_lactate,737734,60.786839
1,median_lactate,737734,60.786839
2,Q25_lactate,737734,60.786839
3,median_neutrophils,563391,46.421553
4,Q75_neutrophils,563391,46.421553
5,Q25_neutrophils,563391,46.421553
6,Q25_lymphocytes,562501,46.348220
7,Q75_lymphocytes,562501,46.348220
8,median_lymphocytes,562501,46.348220
9,Q75_bun,82721,6.815937



📊 Rows per Patient Summary


...

,n_rows,n_columns,n_patients,n_encounters,n_unique_cultures,target_col,prevalence_resistant,missing_features_gt_50pct,missing_features_gt_80pct
0,1213641,71,997,1929,1964,susceptibility,NaN,3,0



📊 Patient / Encounter Counts


,n_patients,n_encounters,patient_col,encounter_col
0,997,1929,anon_id,pat_enc_csn_id_coded



✅ Files saved to: statistics


,n_rows,n_columns,n_patients,n_encounters,n_unique_cultures,target_col,prevalence_resistant,missing_features_gt_50pct,missing_features_gt_80pct
0,1213641,71,997,1929,1964,susceptibility,NaN,3,0
